# 1. Alinha a escala da visão com o Ground Truth e salva a nova trajetória corrigida
evo_traj tum /home/aki/Desktop/GitHub/Python-VO/results/cusco_dataset_1/cusco_dataset_1_vo_SIFT_FLANN_nopnp.txt --ref /home/aki/Desktop/GitHub/Python-VO/results/cusco_dataset_1/cusco_dataset_1_gt_SIFT_FLANN_nopnp.txt -a -s --save_as_tum

# 2. Avalia todo mundo junto sem aplicar nova escala para ser justo com as rodas
evo_ape kitti home/aki/Desktop/GitHub/Python-VO/results/cusco_dataset_1/cusco_dataset_1_gt_SIFT_FLANN_nopnp.txt cusco_dataset_1_gt_SIFT_FLANN_nopnp.kitti.txt -a --plot

In [22]:
import os
import subprocess
import re
import pandas as pd
from IPython.display import Image, display

# ==========================================
# 1. Configurações Iniciais
# ==========================================
# Caminho onde os seus arquivos .txt (formato TUM) estão salvos
BASE_DIR = "/home/aki/Desktop/GitHub/Python-VO/results"

# Definição dos testes que você rodou
datasets = ["00"] 
descriptors = ["ORB"]
matchers = ["KNN"]
suffixes = ["nopnp"] # Ex: "", "ba", "nopnp"

# ==========================================
# 2. Funções Auxiliares para o evo
# ==========================================
def extract_rmse_from_ape(gt_path, est_path, apply_scale=True):
    """Roda o evo_ape e extrai o valor do RMSE (ATE) via Regex."""
    if not os.path.exists(est_path):
        return None
        
    cmd = ["evo_ape", "tum", gt_path, est_path, "-a"]
    if apply_scale:
        cmd.append("-s") # Adiciona alinhamento de escala (Sim(3))
        
    # Executa o comando em background capturando o texto do terminal
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    # Busca o padrão numérico do RMSE no texto de saída do evo
    match = re.search(r"rmse\s+(\d+\.\d+)", result.stdout)
    if match:
        return float(match.group(1))
    return None

def plot_combined_trajectories(gt_path, est_dict, output_name):
    """
    Usa o evo_traj para plotar múltiplas trajetórias na mesma imagem,
    todas alinhadas ao Ground Truth.
    """
    arquivo_plot = f"{output_name}.png"
    
    # overwrite
    if os.path.exists(arquivo_plot):
        os.remove(arquivo_plot)
    # Prepara o comando base
    cmd = ["evo_traj", "tum"]
    
    # Adiciona os arquivos que serão avaliados
    for est_path in est_dict.values():
        cmd.append(est_path)
        
    # Adiciona as opções de referência, alinhamento e nomenclatura
    cmd.extend([
        "--ref", gt_path,
        "-a", "-s", 
        "-p", "--plot_mode", "xz", # Vista superior padrão de robótica
        "--save_plot", f"{output_name}.png"
    ])
        
    print(f"Gerando gráfico comparativo: {output_name}.png ...")
    print(f"Comando executado: {' '.join(cmd)}")
    subprocess.run(cmd)

# ==========================================
# 3. Pipeline Principal de Avaliação
# ==========================================
print("Iniciando avaliação em lote com o pacote EVO...\n")
resultados = []

for dataset in datasets:
    print(f"{'='*50}\nAnalisando Dataset: {dataset}\n{'='*50}")
    work_dir = os.path.join(BASE_DIR, dataset)
    
    # Dicionário para guardar as trajetórias que irão para o gráfico final deste dataset
    trajetorias_para_plot = {}
    
    for desc in descriptors:
        for matcher in matchers:
            for suffix in suffixes:
                suf_str = f"_{suffix}" if suffix else ""
                
                # Nomes esperados dos arquivos
                gt_file = os.path.join(work_dir, f"{dataset}_gt_{desc}_{matcher}{suf_str}.txt")
                vo_file = os.path.join(work_dir, f"{dataset}_vo_{desc}_{matcher}{suf_str}.txt")
                wo_file = os.path.join(work_dir, f"{dataset}_wo.txt") # Assumindo um padrão para a roda
                ba_file = os.path.join(work_dir, f"{dataset}_ba_{desc}_{matcher}{suf_str}.txt")
                
                # Pula se não houver arquivo GT ou VO
                if not os.path.exists(gt_file) or not os.path.exists(vo_file):
                    print("Arquivo nao encontrado", gt_file, vo_file)
                    continue
                
                nome_config = f"{desc} + {matcher}"
                trajetorias_para_plot[f"VO ({nome_config})"] = vo_file
                
                print(f"Avaliando Odometria Visual: {nome_config}")
                ate_vo = extract_rmse_from_ape(gt_file, vo_file, apply_scale=True)
                
                # Verifica se a Odometria de Roda (WO) existe
                ate_wo = "N/A"
                if "kitti" not in dataset.lower() and os.path.exists(wo_file):
                    print("Avaliando Odometria de Roda (WO)...")
                    # Roda geralmente não precisa de escala matemática (-s), pois já tem métrica real
                    ate_wo = extract_rmse_from_ape(gt_file, wo_file, apply_scale=False) 
                    trajetorias_para_plot["Wheel Odometry"] = wo_file
                elif "kitti" in dataset.lower():
                    print("Dataset KITTI detectado: Pulando avaliação de WO.")
                
                # Verifica se houve uso de Bundle Adjustment (BA)
                ate_ba = "N/A"
                if os.path.exists(ba_file):
                    print("Avaliando Bundle Adjustment (BA)...")
                    ate_ba = extract_rmse_from_ape(gt_file, ba_file, apply_scale=True)
                    trajetorias_para_plot[f"BA ({nome_config})"] = ba_file

                resultados.append({
                    "Dataset": dataset,
                    "Configuração": nome_config,
                    "ATE VO (m)": round(ate_vo, 4) if ate_vo else None,
                    "ATE WO (m)": round(ate_wo, 4) if isinstance(ate_wo, float) else ate_wo,
                    "ATE BA (m)": round(ate_ba, 4) if isinstance(ate_ba, float) else ate_ba
                })

    # Gera o gráfico combinando todas as rodadas válidas deste dataset
    if trajetorias_para_plot:
        plot_name = os.path.join(work_dir, f"comparativo_trajetorias_{dataset}")
        plot_combined_trajectories(gt_file, trajetorias_para_plot, plot_name)
        

        if os.path.exists(plot_name + ".png"):
            display(Image(filename=plot_name + ".png"))

# ==========================================
# 4. Ranking e Tabela Final
# ==========================================
print("\n" + "="*60)
print("RANKING FINAL DE DESEMPENHO (Baseado no evo_ape)")
print("="*60)

df_resultados = pd.DataFrame(resultados)
df_resultados.dropna(subset=['ATE VO (m)'], inplace=True) # Remove entradas inválidas
df_resultados = df_resultados.sort_values(by=["Dataset", "ATE VO (m)"])

print(df_resultados.to_string(index=False))

# Opcional: Salva a tabela em um CSV para colocar no texto do TCC
df_resultados.to_csv(os.path.join(BASE_DIR, "tabela_resultados_finais.csv"), index=False)
print(f"\nTabela exportada para: {os.path.join(BASE_DIR, 'tabela_resultados_finais.csv')}")

Iniciando avaliação em lote com o pacote EVO...

Analisando Dataset: 00
Avaliando Odometria Visual: ORB + KNN
Gerando gráfico comparativo: /home/aki/Desktop/GitHub/Python-VO/results/00/comparativo_trajetorias_00.png ...
Comando executado: evo_traj tum /home/aki/Desktop/GitHub/Python-VO/results/00/00_vo_ORB_KNN_nopnp.txt --ref /home/aki/Desktop/GitHub/Python-VO/results/00/00_gt_ORB_KNN_nopnp.txt -a -s -p --plot_mode xz --save_plot /home/aki/Desktop/GitHub/Python-VO/results/00/comparativo_trajetorias_00.png
--------------------------------------------------------------------------------
name:	00_vo_ORB_KNN_nopnp
infos:	46 poses, 43.125m path length, 4.665s duration
--------------------------------------------------------------------------------
name:	00_gt_ORB_KNN_nopnp
infos:	46 poses, 41.652m path length, 4.665s duration
--------------------------------------------------------------------------------
Plot saved to /home/aki/Desktop/GitHub/Python-VO/results/00/comparativo_trajetorias_00